# Day 5 — HOL 2: CDC from PostgreSQL via WAL (Logical Replication)
### GlobalMart Data Engineering · Supabase → Databricks Bronze

---

## What We Are Building

```
Supabase PostgreSQL
  └── poc_orders table
        │
        │  WAL (Write-Ahead Log)
        │  Replication Slot  ← bookmark that records every change
        │
        ▼
  Databricks (JDBC)
        │  pg_logical_slot_get_changes()
        │
        ▼
  bronze/supabase/poc_orders  (Delta table)
```

## HOL 1 vs HOL 2 — The Difference

| | HOL 1 (Watermark) | HOL 2 (WAL / CDC) |
|---|---|---|
| **Captures new rows** | ✅ YES | ✅ YES |
| **Captures updates** | ✅ YES (if `updated_at` changes) | ✅ YES |
| **Captures deletes** | ❌ NO — row is gone, no trace | ✅ YES — WAL records the DELETE |
| **Requires `updated_at` column** | YES — won't work without it | NO — works on any table |
| **Complexity** | Low | Medium |
| **Use when** | Source has `updated_at`, no hard deletes | Full fidelity needed |

## By the End of This HOL You Will Have
- `poc_orders` loaded into Bronze as a full Day 1 snapshot
- A replication slot tracking every change after the snapshot
- CDC events (INSERT + UPDATE + DELETE) captured from the WAL
- A clear mental model of how WAL-based CDC differs from watermark ingestion

---
## The Problem: Why Watermarks Miss Deletes

```
DAY 1 — Source table: poc_orders
  OR-00001  | CUST-10001 | 2026-06-01  ← row exists
  OR-00002  | CUST-10002 | 2026-06-01  ← row exists
  OR-00003  | CUST-10003 | 2026-06-01  ← row exists

Watermark ingestion reads all 3 rows → Bronze has 3 rows ✓

DAY 2 — Customer cancels OR-00003 → row is DELETED from source
  OR-00001  | CUST-10001 | 2026-06-01
  OR-00002  | CUST-10002 | 2026-06-01

Watermark ingestion (WHERE updated_at > last_watermark):
  → No rows returned (nothing was updated, a row was deleted)
  → Bronze still has 3 rows ← OR-00003 is a ghost!

WAL / CDC ingestion:
  → Replication slot recorded: DELETE OR-00003
  → Bronze gets the DELETE event → Silver can apply it
  → Truth preserved ✓
```

### How PostgreSQL WAL Works

```
Every INSERT / UPDATE / DELETE in Postgres is first written to the
Write-Ahead Log (WAL) — before the table is updated.

A logical replication slot is a bookmark in the WAL.
  → It remembers the position of the last change you read.
  → Next read picks up exactly where you left off — no duplicates, no gaps.

pg_logical_slot_get_changes('slot_name', NULL, NULL)
  → Returns all new WAL entries since your last read.
  → DRAINS the slot — run it again and you get nothing until the next change.

This is what Debezium, Fivetran, and Airbyte all use under the hood.
```

In [ ]:
# ─── SETUP ──────────────────────────────────────────────────────────────────────
# Unity Catalog external location handles ADLS auth — no storage key needed.
# Replace YOUR_SUPABASE_PASSWORD before running.

from pyspark.sql.functions import current_timestamp, lit, col, regexp_extract

# ── ADLS / Bronze paths ───────────────────────────────────────────────────────
storage_account  = "amazonprojectadls"
container        = "amazon-data"
base_path        = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
bronze_poc_orders = f"{base_path}/bronze/supabase/poc_orders"

# ── Supabase credentials ──────────────────────────────────────────────────────
# WARNING: Do NOT push this notebook to GitHub with the real password
# Project: amazondata_superbase (qcdequqeczifppyypzhs) — Sydney region
pg_host     = "db.qcdequqeczifppyypzhs.supabase.co"
pg_database = "postgres"
pg_user     = "postgres"
pg_password = "YOUR_SUPABASE_PASSWORD"   # ← paste your password here
jdbc_url    = f"jdbc:postgresql://{pg_host}:5432/{pg_database}?sslmode=require"

def jdbc_read(query):
    return (
        spark.read
        .format("jdbc")
        .option("url",      jdbc_url)
        .option("dbtable",  query)
        .option("user",     pg_user)
        .option("password", pg_password)
        .option("driver",   "org.postgresql.Driver")
        .load()
    )

print(f"Host        : {pg_host}")
print(f"Bronze path : {bronze_poc_orders}")

---
## Pre-Work A — Create Table + Load Day 1 Data in Supabase

> Run these in the **Supabase SQL Editor** for project `qcdequqeczifppyypzhs`.
> Do this **before running any Databricks cells**.

### Step 1 — Create the `poc_orders` table

```sql
CREATE TABLE IF NOT EXISTS public.poc_orders (
    order_id                TEXT PRIMARY KEY,
    customer_id             TEXT NOT NULL,
    order_date              TIMESTAMPTZ,
    shipping_date           TIMESTAMPTZ,
    expected_delivery_date  TIMESTAMPTZ,
    actual_delivery_date    TIMESTAMPTZ,
    shipping_tier_id        TEXT,
    supplier_id             TEXT,
    order_channel           TEXT
);
```

> Notice: **no `updated_at` column** — this is intentional.
> Watermark ingestion won't work on this table. That's exactly why we need CDC.

---

### Step 2 — Load Day 1 data (200 rows)

**Option A — Supabase Table Editor (recommended for this HOL):**
```
Supabase → Table Editor → poc_orders → Import CSV
Upload: globalmart_v2/poc_cdc/orders_sample.csv  (200 rows)
```

**Option B — Run locally (Python):**
```bash
cd globalmart_v2/poc_cdc
cp .env.example .env          # fill in your Supabase values
pip install psycopg2-binary pandas python-dotenv
python 02_load_orders_sample.py
```

**Verify 200 rows loaded:**
```sql
SELECT COUNT(*) FROM public.poc_orders;
-- Expected: 200
```

---
## Pre-Work B — Enable CDC (Logical Replication Slot)

> **CRITICAL TIMING**: The replication slot must be created **AFTER** the initial 200-row load
> and **BEFORE** any Day 2 changes.
>
> If you create the slot before loading Day 1, the initial 200 inserts flood the WAL.
> If you create it after Day 2 changes, you miss them.

Run this in **Supabase SQL Editor** — after the 200 rows are loaded, before Day 2:

```sql
-- 1. Confirm logical replication is enabled (Supabase enables this by default)
SHOW wal_level;
-- Expected: logical

-- 2. Create a publication — declares which table to track
CREATE PUBLICATION poc_orders_pub FOR TABLE public.poc_orders;

-- 3. Create the replication slot — this is the bookmark Databricks will read from
--    test_decoding = built-in plugin, outputs human-readable text
SELECT pg_create_logical_replication_slot('poc_orders_slot', 'test_decoding');

-- 4. Verify the slot exists
SELECT slot_name, plugin, slot_type, active
FROM pg_replication_slots;
-- Expected: poc_orders_slot | test_decoding | logical | false
```

---

### What is `test_decoding`?

```
PostgreSQL ships with two logical decoding plugins:

  test_decoding  → outputs human-readable text
    'table public.poc_orders: INSERT: order_id[text]:"OR-00201" customer_id[text]:"CUST-10001"'
    Good for: learning, debugging, quick POCs

  wal2json       → outputs structured JSON (must be installed separately)
    {"kind":"insert","schema":"public","table":"poc_orders","columnnames":[...],...}
    Good for: production pipelines (easier to parse)

Supabase free tier includes test_decoding.
For production, use Debezium or the Supabase Realtime CDC feature.
```

In [ ]:
# ─── Step 1: Verify JDBC connection + table exists ───────────────────────────────

test_df = jdbc_read("(SELECT COUNT(*) AS total_orders FROM public.poc_orders) t")
test_df.show()

# Also confirm the replication slot was created
slot_df = jdbc_read(
    "(SELECT slot_name, plugin, active FROM pg_replication_slots WHERE slot_name = 'poc_orders_slot') t"
)
slot_df.show(truncate=False)

print("JDBC connection OK. If slot_df is empty — run Pre-Work B first.")

---
## Day 1 — Initial Load → Bronze (Full Snapshot)

```
Initial load strategy:

  1. Read ALL 200 rows from poc_orders via JDBC  (SELECT *)
  2. Tag each row with audit columns
  3. Write to Bronze as a full snapshot (_load_type = 'initial_load')

After this cell runs:
  Bronze has 200 rows
  Replication slot has 0 pending events (it was created after the initial load)

Any change to poc_orders AFTER this point will show up in the slot.
```

In [ ]:
# ─── Step 2: Initial load → Bronze ──────────────────────────────────────────────

initial_df = (
    jdbc_read("(SELECT * FROM public.poc_orders) t")
    .withColumn("_ingested_at",   current_timestamp())
    .withColumn("_source_system", lit("supabase_jdbc"))
    .withColumn("_load_type",     lit("initial_load"))
    .withColumn("_cdc_op",        lit(None).cast("string"))  # NULL for initial load
)

initial_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_poc_orders)

print(f"Initial load complete.")

In [ ]:
# ─── Step 3: Verify Bronze — Day 1 snapshot ─────────────────────────────────────

bronze_df = spark.read.format("delta").load(bronze_poc_orders)

print(f"Bronze poc_orders row count: {bronze_df.count()}")
print()
print("Load type breakdown:")
bronze_df.groupBy("_load_type").count().show()
print()
print("Sample rows:")
bronze_df.select(
    "order_id", "customer_id", "order_channel", "_load_type", "_ingested_at"
).show(5, truncate=False)

---
## PAUSE — Simulate Day 2 Business Activity

> **Before running the CDC capture cell, you must make changes to the live table.**
> Run this SQL in the **Supabase SQL Editor** to simulate Day 2.

```sql
-- ─── Day 2: 5 NEW orders arrive (INSERT) ────────────────────────────────────
INSERT INTO public.poc_orders
    (order_id, customer_id, order_date, shipping_tier_id, order_channel)
VALUES
    ('OR-90001', 'CUST-20001', NOW(), 'STD', 'web'),
    ('OR-90002', 'CUST-20002', NOW(), 'EXP', 'mobile'),
    ('OR-90003', 'CUST-20003', NOW(), 'STD', 'app'),
    ('OR-90004', 'CUST-20004', NOW(), 'PRIME', 'web'),
    ('OR-90005', 'CUST-20005', NOW(), 'STD', 'store');

-- ─── Day 2: Customer upgraded shipping on 2 orders (UPDATE) ─────────────────
-- Replace OR-00001, OR-00002 with actual order IDs from your poc_orders table
UPDATE public.poc_orders
    SET shipping_tier_id = 'EXPRESS'
    WHERE order_id = (SELECT order_id FROM public.poc_orders ORDER BY order_id LIMIT 1);

UPDATE public.poc_orders
    SET order_channel = 'mobile'
    WHERE order_id = (SELECT order_id FROM public.poc_orders ORDER BY order_id OFFSET 1 LIMIT 1);

-- ─── Day 2: Customer cancels order (DELETE) ──────────────────────────────────
-- Watermark ingestion would NEVER see this. CDC captures it.
DELETE FROM public.poc_orders
    WHERE order_id = (SELECT order_id FROM public.poc_orders ORDER BY order_id OFFSET 2 LIMIT 1);

-- Verify: source table should now have 200 + 5 - 1 = 204 rows
SELECT COUNT(*) FROM public.poc_orders;
```

---

### What the replication slot captures

```
After these 8 statements (5 INSERTs + 2 UPDATEs + 1 DELETE), the slot holds:

  BEGIN  <xid>
  table public.poc_orders: INSERT: order_id[text]:'OR-90001' ...
  COMMIT <xid>

  BEGIN  <xid>
  table public.poc_orders: INSERT: order_id[text]:'OR-90002' ...
  COMMIT <xid>
  ... (5 INSERTs total)

  BEGIN  <xid>
  table public.poc_orders: UPDATE: order_id[text]:'OR-000XX' ... shipping_tier_id[text]:'EXPRESS'
  COMMIT <xid>

  BEGIN  <xid>
  table public.poc_orders: DELETE: order_id[text]:'OR-000XX'
  COMMIT <xid>

Total rows from pg_logical_slot_get_changes: 8 statements × 3 rows (BEGIN + event + COMMIT) = 24 rows
```

> **Run the SQL above in Supabase before continuing.**

---
## CDC Capture — Reading the Replication Slot

```
pg_logical_slot_get_changes('poc_orders_slot', NULL, NULL)

  Parameters:
    slot_name  = 'poc_orders_slot'
    upto_lsn   = NULL  → drain everything up to the current WAL position
    upto_nchanges = NULL  → no limit on number of changes

  Returns 3 columns:
    lsn     = Log Sequence Number (WAL position)
    xid     = Transaction ID
    data    = The human-readable change text

  ⚠️  IMPORTANT: This function DRAINS the slot.
    Run 1 → returns 24 rows (all Day 2 changes)
    Run 2 → returns 0 rows (already drained)
    The slot moves forward. Old events are gone.

  That's why we .cache() the result — so .count() and .show() read the same data
  without draining the slot twice.
```

In [ ]:
# ─── Step 4: Capture CDC events from the replication slot ───────────────────────
# .cache() is critical — pg_logical_slot_get_changes() drains on first read.
# Without cache, count() drains it and show() returns empty.

cdc_query = (
    "(SELECT * FROM pg_logical_slot_get_changes('poc_orders_slot', NULL, NULL)) AS cdc_events"
)

cdc_raw_df = jdbc_read(cdc_query).cache()

total_wal_rows = cdc_raw_df.count()
print(f"Total WAL rows captured: {total_wal_rows}")
print()
print("(Each statement = 3 WAL rows: BEGIN + event + COMMIT)")
print(f"Estimated statements: {total_wal_rows // 3}")
print()
print("If this shows 0: either Day 2 SQL hasn't been run yet, or the slot was already drained.")

In [ ]:
# ─── Step 5: Inspect raw CDC events ─────────────────────────────────────────────
# Look at the raw WAL text to understand the format.

print("Raw WAL output — all rows including BEGIN and COMMIT markers:")
cdc_raw_df.show(30, truncate=False)

In [ ]:
# ─── Step 6: Parse event types ──────────────────────────────────────────────────
# Filter out BEGIN/COMMIT noise — keep only the actual data change rows.
# Extract the operation type (INSERT / UPDATE / DELETE) from the data column.

data_events_df = (
    cdc_raw_df
    .filter(
        col("data").startswith("table ")  # real events start with "table <schema>.<table>:"
    )
    .withColumn(
        "_cdc_op",
        regexp_extract(col("data"), r"table public\.poc_orders: (INSERT|UPDATE|DELETE):", 1)
    )
    .withColumn(
        "_order_id",
        regexp_extract(col("data"), r"order_id\[text\]:'([^']+)'", 1)
    )
    .withColumn("_captured_at", current_timestamp())
    .select("lsn", "xid", "_cdc_op", "_order_id", "_captured_at", "data")
)

print("CDC events by operation type:")
data_events_df.groupBy("_cdc_op").count().show()

print("\nParsed events:")
data_events_df.select("_cdc_op", "_order_id", "_captured_at").show(20, truncate=False)

In [ ]:
# ─── Step 7: Append CDC events to Bronze ────────────────────────────────────────
# Bronze is append-only — CDC events land as new rows with _cdc_op column.
# Silver will later MERGE these events to maintain the current state.

bronze_cdc_df = (
    data_events_df
    .withColumn("_ingested_at",   current_timestamp())
    .withColumn("_source_system", lit("supabase_cdc"))
    .withColumn("_load_type",     lit("cdc_event"))
    .select(
        "_order_id",
        "_cdc_op",
        "lsn",
        "xid",
        "data",
        "_ingested_at",
        "_source_system",
        "_load_type"
    )
)

bronze_cdc_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .save(bronze_poc_orders)

total = spark.read.format("delta").load(bronze_poc_orders).count()
print(f"Bronze poc_orders total rows: {total}")
print()

print("Breakdown by load_type:")
spark.read.format("delta").load(bronze_poc_orders) \
    .groupBy("_load_type", "_cdc_op").count() \
    .orderBy("_load_type") \
    .show()

---
## What Bronze Looks Like Now

```
Bronze poc_orders after this HOL:

  _load_type       _cdc_op    rows
  ────────────── ─ ────────── ────
  initial_load     NULL        200   ← Day 1 snapshot
  cdc_event        INSERT        5   ← 5 new Day 2 orders
  cdc_event        UPDATE        2   ← 2 shipping upgrades
  cdc_event        DELETE        1   ← 1 cancelled order
  ─────────────────────────────────
  TOTAL                         208

This is RAW Bronze — it captures everything including contradictions:
  A row can appear in initial_load AND then in a DELETE event.
  Bronze doesn't resolve this — Silver does.
```

## What Silver Will Do With This (Next Session)

```
Silver reads Bronze CDF events and applies MERGE logic:

  CDC INSERT  → MERGE INTO silver WHEN NOT MATCHED THEN INSERT
  CDC UPDATE  → MERGE INTO silver WHEN MATCHED THEN UPDATE
  CDC DELETE  → MERGE INTO silver WHEN MATCHED THEN DELETE
                (or: SET is_deleted = true for soft delete)

Result: silver.poc_orders always reflects the CURRENT state of the source.
  After Day 2:
    Silver has 204 rows  (200 initial + 5 new - 1 deleted)
    2 rows have updated shipping_tier_id
    1 row is gone (or marked is_deleted = true)
```

---

## Key Concepts to Remember

| Concept | What It Means |
|---|---|
| **Replication slot** | A bookmark in the WAL — remembers where you last read |
| **test_decoding** | Built-in plugin — outputs human-readable text (not JSON) |
| **Drain behaviour** | `pg_logical_slot_get_changes()` advances the bookmark — re-run = 0 rows |
| **`.cache()`** | Materialises the result before Spark makes a second JDBC call |
| **BEGIN / COMMIT rows** | WAL wraps every statement — filter them out before processing |
| **Bronze append-only** | CDC events land as new rows — Silver resolves the final state |

---

## Discussion Questions

1. *The replication slot was created AFTER the Day 1 data was loaded. What would happen if it was created BEFORE the 200-row load?*

2. *`pg_logical_slot_get_changes()` drains the slot. What happens if Databricks crashes mid-read? How would you recover?*

3. *The raw `data` column is text. In production, which plugin would you use instead of `test_decoding` for easier parsing, and why?*

4. *Bronze has both the original row (initial_load) AND a DELETE event for the same `order_id`. How does Silver know which one is "truth"?*

5. *We used `mode("overwrite")` for the initial load but `mode("append")` for CDC events. Why the difference?*